# Benchmarking Open-Source LLMs for Triton Kernel Generation

**Target runtime:** Google Colab A100 40 GB  
**Backend:** Triton  
**Benchmark levels:** Level 1 (single-kernel ops) & Level 2 (fusion patterns)

---

## What you will learn

1. **KernelBench** – how GPU-kernel benchmarks are structured and evaluated.
2. **Triton** – the anatomy of a `@triton.jit` kernel and how it differs from raw PyTorch.
3. **vLLM** – how to run open-source LLMs locally with hardware-efficient inference.
4. **Prompt engineering** – how KernelBench constructs prompts that guide an LLM to write kernels.
5. **Evaluation pipeline** – correctness checking, performance measurement against `torch.compile`, and a custom _Validity_ metric.
6. **Cross-model comparison** – visualising which models write better GPU kernels.

---

### Three benchmark configurations across two model weights

| Tag | Model | HuggingFace ID | VRAM | Thinking |
|-----|-------|---------------|------|---------|
| `Qwen2.5-Coder-7B` | Qwen2.5-Coder-7B-Instruct | `Qwen/Qwen2.5-Coder-7B-Instruct` | ~16 GB (bf16) | ❌ |
| `Qwen3-8B-think` | Qwen3-8B | `Qwen/Qwen3-8B` | ~16 GB (bf16) | ✅ `temp=0.6` |
| `Qwen3-8B-no-think` | Qwen3-8B | `Qwen/Qwen3-8B` | ~16 GB (bf16) | ❌ `temp=0.7` |

Qwen3-8B is loaded **once** and reused for both thinking and non-thinking runs.

### Research questions this benchmark can answer

|  | No thinking | Thinking |
|--|-------------|---------|
| **Code-specialized** | Qwen2.5-Coder-7B | — |
| **General-purpose** | Qwen3-8B-no-think | Qwen3-8B-think |

- Does **code specialisation** help for Triton kernel generation?
- Does **chain-of-thought reasoning** (thinking mode) help?
- How much does thinking mode help Qwen3-8B specifically? (`Qwen3-8B-think` vs `Qwen3-8B-no-think`)

---
## Section 0 — Environment Check

First, confirm you are running on an A100 GPU and that CUDA is available.

In [3]:
!nvidia-smi

zsh:1: command not found: nvidia-smi


In [4]:
import torch
assert torch.cuda.is_available(), "No GPU detected! Switch runtime to GPU (A100) in Colab."
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"Device   : {torch.cuda.get_device_name(0)}")
print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

AssertionError: No GPU detected! Switch runtime to GPU (A100) in Colab.

---
## Section 1 — Install Dependencies

We need three things:
- **KernelBench** – the benchmark framework (cloned from GitHub).
- **vLLM** – fast open-source LLM inference engine.
- **Triton** – already bundled with recent PyTorch; we verify it here.

In [5]:
# Clone KernelBench (skip if already cloned)
import os
if not os.path.isdir("KernelBench"):
    !git clone https://github.com/ScalingIntelligence/KernelBench.git
else:
    print("KernelBench already cloned.")

KernelBench already cloned.


In [1]:
# Install KernelBench as a Python package (editable) and its core deps
from pathlib import Path

kernelbench_candidates = [Path("KernelBench"), Path("../KernelBench")]
KERNELBENCH_CHECKOUT = next(
    (
        p for p in kernelbench_candidates
        if (p / "scripts" / "eval_from_generations.py").exists()
        and (p / "src" / "kernelbench").exists()
    ),
    None,
)
if KERNELBENCH_CHECKOUT is None:
    raise FileNotFoundError(
        "Could not find a full KernelBench checkout. Clone it at ./KernelBench "
        "or ../KernelBench, or set KERNELBENCH_ROOT before running this notebook."
    )

!pip install -q -e {KERNELBENCH_CHECKOUT}
# Install vLLM (choose a wheel compatible with your CUDA version)
!pip install -q vllm
# Other small helpers used in this notebook
!pip install -q tqdm matplotlib seaborn pandas

print(f"Installed KernelBench from {KERNELBENCH_CHECKOUT}")


Installed KernelBench from KernelBench


In [2]:
# Verify all key imports
import os
import sys
from pathlib import Path

kernelbench_root = Path(os.getenv("KERNELBENCH_ROOT", "KernelBench"))
if not (kernelbench_root / "src" / "kernelbench").exists():
    fallback_root = Path("../KernelBench")
    if (fallback_root / "src" / "kernelbench").exists():
        kernelbench_root = fallback_root

src_path = str(kernelbench_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)   # make kernelbench importable

import triton
import triton.language as tl
from vllm import LLM, SamplingParams
import kernelbench

print(f"KernelBench root: {kernelbench_root}")
print(f"Triton   : {triton.__version__}")
print(f"vLLM     : imported OK")
print(f"KernelBench: imported OK")


ModuleNotFoundError: No module named 'triton'

---
## Section 2 — KernelBench Fundamentals

KernelBench is organised into **levels of difficulty**:

| Level | Description | Count |
|-------|-------------|-------|
| **1** | Single-kernel ops (ReLU, matmul, layer norm …) | 100 |
| **2** | Simple fusion patterns (Conv + Bias + ReLU, Gemm + Scale + LeakyReLU …) | 100 |
| **3** | Full model architectures | 50 |

Each problem is a Python file that defines:
- `Model` – the reference PyTorch implementation.
- `get_inputs()` – generates random tensors for each forward pass.
- `get_init_inputs()` – constructor arguments for `Model`.

The LLM's task: replace `Model` with `ModelNew` that implements the same computation using a Triton kernel.

In [6]:
from kernelbench.dataset import construct_kernelbench_dataset

# Load Level 1 and Level 2 datasets from HuggingFace
dataset_l1 = construct_kernelbench_dataset(level=1, source="huggingface")
dataset_l2 = construct_kernelbench_dataset(level=2, source="huggingface")

print(f"Level 1 problems: {len(dataset_l1)}")
print(f"Level 2 problems: {len(dataset_l2)}")


AttributeError: partially initialized module 'litellm' has no attribute 'litellm_core_utils' (most likely due to a circular import)

In [ ]:
# Inspect the first Level 1 problem (KernelBench problem IDs are 1-indexed)
prob_l1 = dataset_l1.get_problem_by_id(1)
print(f"Problem name : {prob_l1.name}")
print(f"Level        : {prob_l1.level}")
print("-" * 60)
print(prob_l1.code)


In [ ]:
# Inspect a Level 2 problem to see the difference (fusion pattern)
prob_l2 = dataset_l2.get_problem_by_id(12)  # e.g. Gemm_Multiply_LeakyReLU
print(f"Problem name : {prob_l2.name}")
print(f"Level        : {prob_l2.level}")
print("-" * 60)
print(prob_l2.code)


### Key observations

- **Level 1** is a single operation — the LLM just needs one well-written Triton kernel.
- **Level 2** chains multiple ops (GEMM → multiply → LeakyReLU). A fused Triton kernel can be _faster_ than calling three separate ops, which is why fusion is interesting.

The LLM never sees the ground-truth `ModelNew`; it must write one from scratch.

---
## Section 3 — Introduction to Triton

Triton is a Python DSL for writing GPU kernels.  
Unlike raw CUDA C++, Triton works at the **tile / block level**, not the individual thread level, which makes it much easier to write while still achieving near-peak hardware utilisation.

### The building blocks

| Concept | Triton equivalent | What it does |
|---------|------------------|--------------|
| Thread block | `tl.program_id(axis)` | ID of the current block in a given axis |
| Block of elements | `tl.arange(0, BLOCK_SIZE)` | A range of indices |
| Bounds check | `mask = offsets < n_elements` | Prevents out-of-bounds access |
| Load | `tl.load(ptr + offsets, mask=mask)` | Read a block of memory |
| Store | `tl.store(ptr + offsets, value, mask=mask)` | Write a block of memory |

The `@triton.jit` decorator **compiles** the function to PTX the first time it runs.

In [ ]:
# The canonical KernelBench Triton example: element-wise addition
# This is the exact file in src/kernelbench/prompts/model_new_ex_add_triton.py

import torch
import torch.nn as nn
import triton
import triton.language as tl


# ── Step 1: Define the kernel ──────────────────────────────────────────────
@triton.jit
def add_kernel(
    x_ptr,          # pointer to first input
    y_ptr,          # pointer to second input
    out_ptr,        # pointer to output
    n_elements,     # total number of elements
    BLOCK_SIZE: tl.constexpr,  # compile-time constant (tunable)
):
    # Which block of BLOCK_SIZE elements am I responsible for?
    block_start = tl.program_id(0) * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements      # guard against partial last block

    x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
    y = tl.load(y_ptr + offsets, mask=mask, other=0.0)
    tl.store(out_ptr + offsets, x + y, mask=mask)


# ── Step 2: Python wrapper (handles allocation, grid, launch) ──────────────
def triton_add(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    x, y = x.contiguous(), y.contiguous()
    out = torch.empty_like(x)
    n = x.numel()
    BLOCK_SIZE = 1024
    grid = lambda meta: (triton.cdiv(n, meta["BLOCK_SIZE"]),)
    add_kernel[grid](x, y, out, n, BLOCK_SIZE=BLOCK_SIZE)
    return out


# ── Step 3: Wrap in an nn.Module so KernelBench can evaluate it ───────────
class ModelNew(nn.Module):
    def forward(self, a, b):
        return triton_add(a, b)


print("Triton kernel defined.")

In [ ]:
# Test the kernel against PyTorch
a = torch.randn(1, 128, device="cuda")
b = torch.randn(1, 128, device="cuda")

ref_out  = a + b                             # PyTorch reference
trit_out = triton_add(a, b)                  # Our Triton kernel

print(f"Max absolute difference: {(ref_out - trit_out).abs().max().item():.2e}")
assert torch.allclose(ref_out, trit_out, atol=1e-4), "Outputs don't match!"
print("Correctness check PASSED — Triton kernel matches PyTorch.")

In [ ]:
# Quick timing comparison on a large tensor
import time

N = 10_000_000
a_big = torch.randn(N, device="cuda")
b_big = torch.randn(N, device="cuda")

# Warm-up
for _ in range(5):
    _ = a_big + b_big
    _ = triton_add(a_big, b_big)
torch.cuda.synchronize()

# Time PyTorch
t0 = time.perf_counter()
for _ in range(100):
    _ = a_big + b_big
torch.cuda.synchronize()
t_torch = (time.perf_counter() - t0) / 100 * 1000

# Time Triton
t0 = time.perf_counter()
for _ in range(100):
    _ = triton_add(a_big, b_big)
torch.cuda.synchronize()
t_triton = (time.perf_counter() - t0) / 100 * 1000

print(f"PyTorch add : {t_torch:.3f} ms")
print(f"Triton add  : {t_triton:.3f} ms")
print(f"Speedup     : {t_torch / t_triton:.2f}×")

> **Note:** For a simple element-wise add, Triton and PyTorch are both memory-bandwidth bound — you will see roughly 1× speedup. Triton shines when **fusing** multiple ops to avoid extra memory round-trips.

---
## Section 4 — Prompt Construction

KernelBench constructs prompts automatically via `get_prompt_for_backend()`.  
A typical **one-shot Triton prompt** has four parts:

1. **Task description** — what the LLM is being asked to do.
2. **One-shot example** — the `model_ex_add.py` reference + `model_new_ex_add_triton.py` target, showing the input/output format.
3. **Your problem** — the reference `Model` the LLM must replace.
4. **Instruction** — "Output `ModelNew` only, wrapped in a code block."

In [ ]:
from kernelbench.prompt_constructor_toml import get_prompt_for_backend
import os

# Build a Triton one-shot prompt for the first Level 1 problem
prompt = get_prompt_for_backend(
    ref_arch_src=prob_l1.code,   # the reference PyTorch code
    backend="triton",
    option="one_shot",
)

print(f"Prompt length: {len(prompt)} characters, ~{len(prompt)//4} tokens")


In [ ]:
# Print the full prompt so you can see exactly what the LLM receives
# (truncated for readability; set SHOW_FULL=True to see everything)
SHOW_FULL = False
if SHOW_FULL:
    print(prompt)
else:
    MAX_CHARS = 3000
    print(prompt[:MAX_CHARS])
    if len(prompt) > MAX_CHARS:
        print(f"\n... [{len(prompt) - MAX_CHARS} chars truncated] ...")

---
## Section 5 — Setting Up vLLM for Inference

[vLLM](https://github.com/vllm-project/vllm) is a fast, memory-efficient inference engine for open-source LLMs.  
Key advantages for this notebook:
- **PagedAttention** — serves long prompts efficiently without OOM.
- **Offline mode** — no server subprocess needed; just call `llm.chat()` inside Python.
- **Shared weights** — Qwen3-8B is loaded once and reused for both thinking settings.

### Memory budget on A100 40 GB

| Model | bf16 VRAM | Strategy |
|-------|-----------|----------|
| Qwen2.5-Coder-7B | ~16 GB | Load as bf16, `gpu_memory_utilization=0.85` |
| Qwen3-8B | ~16 GB | Load as bf16, `gpu_memory_utilization=0.85` |

### Sampling parameters differ per model

Qwen3-8B's thinking mode **must not use greedy decoding** — the model card explicitly warns against `temperature=0` when `<think>` mode is active, as it causes degenerate repetitions. We therefore use per-model `SamplingParams`:

| Model | temperature | top_p | top_k | Notes |
|-------|------------|-------|-------|-------|
| Qwen2.5-Coder-7B | 0.0 | — | — | Greedy, deterministic |
| Qwen3-8B-no-think | 0.7 | 0.8 | 20 | Same chat path as think mode, thinking disabled |
| Qwen3-8B-think | 0.6 | 0.95 | 20 | Thinking mode recommended settings |


In [ ]:
# All benchmark configuration lives in MODEL_CONFIGS, keyed by model_tag.
# Two tags share the same model_id (Qwen3-8B) but differ only in the
# thinking flag and sampling parameters — this is the key comparison.
MODEL_CONFIGS = {
    "Qwen2.5-Coder-7B": {
        "model_id":       "Qwen/Qwen2.5-Coder-7B-Instruct",
        "load_kwargs":    {"dtype": "bfloat16", "gpu_memory_utilization": 0.85},
        "sampling_params": SamplingParams(temperature=0.0, max_tokens=4096, seed=0),
        "thinking":       False,
        "chat_template_kwargs": None,
    },
    "Qwen3-8B-think": {
        "model_id":       "Qwen/Qwen3-8B",
        "load_kwargs":    {"dtype": "bfloat16", "gpu_memory_utilization": 0.85},
        # Thinking mode requires non-greedy decoding (model card recommendation)
        "sampling_params": SamplingParams(
            temperature=0.6, top_p=0.95, top_k=20, max_tokens=4096, seed=0
        ),
        "thinking":       True,
        "chat_template_kwargs": {"enable_thinking": True},
    },
    "Qwen3-8B-no-think": {
        "model_id":       "Qwen/Qwen3-8B",
        "load_kwargs":    {"dtype": "bfloat16", "gpu_memory_utilization": 0.85},
        "sampling_params": SamplingParams(
            temperature=0.7, top_p=0.8, top_k=20, max_tokens=4096, seed=0
        ),
        "thinking":       False,
        "chat_template_kwargs": {"enable_thinking": False},
    },
}

print(f"{len(MODEL_CONFIGS)} benchmark configurations defined:")
for tag, cfg in MODEL_CONFIGS.items():
    think_str = "thinking=ON " if cfg["thinking"] else "thinking=OFF"
    temp_str  = f"temp={cfg['sampling_params'].temperature}"
    print(f"  {tag:<22}  {cfg['model_id']:<48}  {think_str}  {temp_str}")


In [ ]:
MODEL_MAX_LEN = 16384


def strip_think_block(text: str) -> str:
    cleaned = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    if "<think>" in cleaned:
        cleaned = re.sub(r"<think>.*", "", cleaned, flags=re.DOTALL).strip()
    return cleaned


def load_vllm_model(model_tag: str) -> LLM:
    """Load the LLM for a given model_tag using settings from MODEL_CONFIGS."""
    cfg    = MODEL_CONFIGS[model_tag]
    kwargs = cfg["load_kwargs"]
    print(f"Loading model: {cfg['model_id']}  (tag={model_tag})")
    print(f"  settings: {kwargs}")
    llm = LLM(
        model=cfg["model_id"],
        max_model_len=MODEL_MAX_LEN,
        trust_remote_code=True,   # required for Qwen models
        **kwargs,
    )
    print("Model loaded.")
    return llm


def unload_vllm_model(llm: LLM) -> None:
    """Delete vLLM state and release GPU memory before loading the next model."""
    import gc

    try:
        engine = getattr(llm, "llm_engine", None)
        engine_core = getattr(engine, "engine_core", None)
        if engine_core is not None and hasattr(engine_core, "shutdown"):
            engine_core.shutdown()
        elif engine_core is not None:
            print("warning: vLLM engine_core found but has no shutdown() method")
        model_executor = getattr(engine, "model_executor", None)
        if model_executor is not None and hasattr(model_executor, "shutdown"):
            model_executor.shutdown()
        elif model_executor is not None:
            print("warning: vLLM model_executor found but has no shutdown() method")
    except Exception as exc:
        print(f"warning: vLLM shutdown helper failed: {exc}")

    for module_name, func_name in [
        ("vllm.distributed.parallel_state", "cleanup_dist_env_and_memory"),
        ("vllm.distributed.parallel_state", "destroy_model_parallel"),
        ("vllm.distributed", "destroy_distributed_environment"),
    ]:
        try:
            module = __import__(module_name, fromlist=[func_name])
            getattr(module, func_name)()
        except Exception as exc:
            print(f"warning: cleanup helper {module_name}.{func_name} failed: {exc}")

    del llm
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        if hasattr(torch.cuda, "ipc_collect"):
            torch.cuda.ipc_collect()
    print("Model unloaded. GPU memory released.")


In [ ]:
# Quick smoke-test: load the first model and ask a trivial question
# (This verifies vLLM is working before the full benchmark)
test_tag = "Qwen2.5-Coder-7B"
llm_test = load_vllm_model(test_tag)

test_prompt = "Write a Python function that returns the sum of two numbers."
test_messages = [{"role": "user", "content": test_prompt}]
test_output = llm_test.chat([test_messages], MODEL_CONFIGS[test_tag]["sampling_params"])
print("=== Model response ===")
print(test_output[0].outputs[0].text[:500])

unload_vllm_model(llm_test)


---
## Section 6 — Generate a Single Kernel & Check Validity

Before running the full benchmark, let's work through one problem end-to-end.

### The Validity metric

The standard KernelBench metrics (correctness, speedup) measure _what_ a kernel does.  
The **Validity** metric measures _how_ it does it:

> A kernel is **Valid** if the generated code contains at least one `@triton.jit`-decorated function **and** that function's body does not call `torch.*` APIs.

A "cheat" kernel that simply wraps `torch.matmul(A, B)` without any Triton code is **Invalid** — even if it produces the correct answer.

In [ ]:
import ast

def check_triton_validity(kernel_code: str) -> dict:
    """
    Check whether a generated kernel is a genuine Triton implementation.

    Returns a dict with:
      has_triton_jit          – bool: at least one @triton.jit function exists
      torch_calls_in_jit      – list[str]: any torch.* calls found inside @triton.jit bodies
      is_valid                – bool: True if has_triton_jit and no torch.* inside kernel
      needs_manual_review     – bool: True when torch calls were found (borderline case)
      notes                   – str: human-readable explanation
    """
    result = {
        "has_triton_jit": False,
        "torch_calls_in_jit": [],
        "is_valid": False,
        "needs_manual_review": False,
        "notes": "",
    }

    if "@triton.jit" not in kernel_code:
        result["notes"] = "No @triton.jit decorator found — not a Triton kernel"
        return result

    result["has_triton_jit"] = True

    try:
        tree = ast.parse(kernel_code)
    except SyntaxError as e:
        result["notes"] = f"SyntaxError while parsing: {e}"
        result["needs_manual_review"] = True
        return result

    class _Visitor(ast.NodeVisitor):
        def __init__(self):
            self.in_jit = False
            self.torch_calls = []

        def visit_FunctionDef(self, node):
            # Check for @triton.jit decorator
            is_jit = any(
                isinstance(d, ast.Attribute)
                and isinstance(d.value, ast.Name)
                and d.value.id == "triton"
                and d.attr == "jit"
                for d in node.decorator_list
            )
            was = self.in_jit
            if is_jit:
                self.in_jit = True
            self.generic_visit(node)
            self.in_jit = was

        def visit_Attribute(self, node):
            if (
                self.in_jit
                and isinstance(node.value, ast.Name)
                and node.value.id == "torch"
            ):
                self.torch_calls.append(ast.unparse(node))
            self.generic_visit(node)

    v = _Visitor()
    v.visit(tree)

    result["torch_calls_in_jit"] = v.torch_calls
    result["is_valid"] = len(v.torch_calls) == 0
    result["needs_manual_review"] = bool(v.torch_calls)
    if result["is_valid"]:
        result["notes"] = "Valid: @triton.jit kernel with no torch.* calls inside the kernel body"
    else:
        result["notes"] = f"Invalid: torch.* calls inside @triton.jit: {v.torch_calls}"
    return result


print("check_triton_validity() defined.")

In [ ]:
# Demonstrate the validity checker on known-good and trivial kernels

# GOOD: genuine Triton kernel
good_kernel = """
import triton, triton.language as tl, torch, torch.nn as nn

@triton.jit
def relu_kernel(x_ptr, out_ptr, n: tl.constexpr, BLOCK: tl.constexpr):
    i = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    mask = i < n
    x = tl.load(x_ptr + i, mask=mask)
    tl.store(out_ptr + i, tl.maximum(x, 0.0), mask=mask)

class ModelNew(nn.Module):
    def forward(self, x):
        out = torch.empty_like(x)
        relu_kernel[(triton.cdiv(x.numel(), 128),)](x, out, x.numel(), BLOCK=128)
        return out
"""

# BAD: no Triton at all — just wraps PyTorch
bad_kernel = """
import torch, torch.nn as nn

class ModelNew(nn.Module):
    def forward(self, x):
        return torch.relu(x)   # no @triton.jit — Invalid!
"""

for label, code in [("GOOD kernel", good_kernel), ("BAD kernel", bad_kernel)]:
    r = check_triton_validity(code)
    print(f"[{label}]")
    print(f"  has_triton_jit      : {r['has_triton_jit']}")
    print(f"  is_valid            : {r['is_valid']}")
    print(f"  needs_manual_review : {r['needs_manual_review']}")
    print(f"  notes               : {r['notes']}")
    print()

In [ ]:
import re
from kernelbench.utils import extract_first_code


def strip_thinking_block(text: str) -> str:
    """Remove <think> blocks emitted by Qwen3, including truncated outputs."""
    cleaned = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    if "<think>" in cleaned:
        cleaned = re.sub(r"<think>.*", "", cleaned, flags=re.DOTALL).strip()
    return cleaned


def generate_triton_kernel(
    llm: LLM,
    model_tag: str,
    problem,
    option: str = "one_shot",
) -> str:
    """
    Build a Triton prompt for `problem`, query `llm`, and extract the code block.

    All instruct models use the chat path. Qwen3 differs only in whether
    `enable_thinking` is passed through `chat_template_kwargs`.

    Returns the extracted Python source code string (empty string on failure).
    """
    cfg             = MODEL_CONFIGS[model_tag]
    sampling_params = cfg["sampling_params"]
    use_thinking    = cfg["thinking"]
    chat_kwargs     = cfg.get("chat_template_kwargs")

    prompt = get_prompt_for_backend(
        ref_arch_src=problem.code,
        backend="triton",
        option=option,
    )

    messages = [{"role": "user", "content": prompt}]
    if chat_kwargs:
        outputs = llm.chat(
            [messages],
            sampling_params=sampling_params,
            chat_template_kwargs=chat_kwargs,
        )
    else:
        outputs = llm.chat([messages], sampling_params=sampling_params)

    raw_text = outputs[0].outputs[0].text
    clean_text = strip_thinking_block(raw_text) if use_thinking else raw_text

    code = extract_first_code(clean_text, ["python", "py"])
    return code if code else ""


print("generate_triton_kernel() defined.")


In [ ]:
# Generate a kernel for the first Level 1 problem with Qwen2.5-Coder-7B
demo_tag = "Qwen2.5-Coder-7B"
llm_demo = load_vllm_model(demo_tag)

demo_problem = dataset_l1.get_problem_by_id(1)
print(f"Generating kernel for: {demo_problem.name}")

generated_kernel = generate_triton_kernel(llm_demo, demo_tag, demo_problem)

print("\n=== Generated kernel ===")
print(generated_kernel[:2000] if len(generated_kernel) > 2000 else generated_kernel)


In [ ]:
# Check the validity of the generated kernel
validity = check_triton_validity(generated_kernel)
print("=== Validity report ===")
for k, v in validity.items():
    print(f"  {k:<25}: {v}")

In [ ]:
# Unload the demo model before moving to evaluation
unload_vllm_model(llm_demo)

---
## Section 7 — Evaluate a Single Kernel

`eval_kernel_against_ref()` is the heart of KernelBench's evaluation pipeline:

1. **Compile** – loads `ModelNew` via a temp-file import (required for `@triton.jit`).
2. **Correctness** – runs 5 independent trials with seeded random inputs and compares outputs to the PyTorch reference with `torch.allclose(atol=1e-4)`.
3. **Performance** – if correct, measures wall-clock time over 100 trials after L2 cache clearing, for both the custom kernel and the reference.

The result is a `KernelExecResult` object.

In [ ]:
from kernelbench.eval import eval_kernel_against_ref

# Evaluate the kernel we just generated
print(f"Evaluating kernel for: {demo_problem.name}")
print("(This may take ~30 s for compilation + 100 timing trials)")

if generated_kernel:
    result = eval_kernel_against_ref(
        original_model_src=demo_problem.code,
        custom_model_src=generated_kernel,
        backend="triton",
        measure_performance=True,
        num_correct_trials=5,
        num_perf_trials=100,
    )

    print("\n=== KernelExecResult ===")
    print(f"  compiled    : {result.compiled}")
    print(f"  correctness : {result.correctness}")
    if result.runtime:
        print(f"  runtime     : {result.runtime:.4f} ms  (kernel)")
        print(f"  ref_runtime : {result.ref_runtime:.4f} ms  (eager PyTorch)")
        speedup_eager = result.ref_runtime / result.runtime
        print(f"  speedup     : {speedup_eager:.2f}× vs eager PyTorch")
    if result.metadata.get("error"):
        print(f"  error       : {result.metadata['error'][:200]}")
else:
    print("No kernel code was generated — skipping eval.")

In [ ]:
# Educational: what happens when a kernel fails correctness?
# Here we craft a deliberately wrong kernel
broken_kernel = """
import torch, torch.nn as nn, triton, triton.language as tl

@triton.jit
def broken_kernel(x_ptr, out_ptr, n: tl.constexpr, BLOCK: tl.constexpr):
    # Bug: uses + instead of max(x, 0)
    i = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    mask = i < n
    x = tl.load(x_ptr + i, mask=mask)
    tl.store(out_ptr + i, x + 1.0, mask=mask)   # wrong! adds 1 instead of ReLU

class ModelNew(nn.Module):
    def forward(self, x):
        out = torch.empty_like(x)
        broken_kernel[(triton.cdiv(x.numel(), 128),)](x, out, x.numel(), BLOCK=128)
        return out
"""

# Use the final Level 1 problem as the reference for this demo
last_l1_pid = max(dataset_l1.get_problem_ids())
r_broken = eval_kernel_against_ref(
    original_model_src=dataset_l1.get_problem_by_id(last_l1_pid).code,
    custom_model_src=broken_kernel,
    backend="triton",
    measure_performance=False,  # skip timing for wrong kernel
    num_correct_trials=5,
)
print(f"compiled    : {r_broken.compiled}")
print(f"correctness : {r_broken.correctness}")
if r_broken.metadata.get("error"):
    print(f"error detail: {r_broken.metadata['error'][:400]}")


---
## Section 8 — The `torch.compile` Baseline

Instead of comparing against **eager PyTorch** (the default `ref_runtime`), we use **`torch.compile` (Inductor backend)** as our baseline.

### Why `torch.compile`?

- `torch.compile` already applies operator fusion, memory layout optimisations, and GPU-specific code generation.
- Beating it is **significantly harder** — and a more meaningful signal that the LLM understood the hardware.
- It is the standard production baseline in modern PyTorch code.

KernelBench ships pre-computed `torch.compile` baseline times for several GPUs:

In [ ]:
import json, os, glob

# Look for pre-computed torch.compile baselines shipped with KernelBench
baseline_glob = str(KERNELBENCH_ROOT / "results" / "timing" / "**" / "baseline_time_torch_compile_inductor_default.json")
baseline_files = glob.glob(baseline_glob, recursive=True)
print("Available torch.compile baseline files:")
for f in baseline_files:
    print(f"  {f}")

In [ ]:
def load_baseline(baseline_file: str) -> dict:
    """Load a KernelBench baseline timing file."""
    with open(baseline_file) as f:
        data = json.load(f)
    return data


def get_compile_baseline_for_problem(baseline_data: dict, problem, fallback_runtime: float = None):
    """
    Look up the torch.compile baseline runtime (ms) for a given problem.

    Returns None if not found (will fall back to eager runtime in that case).
    """
    level_key = f"level{problem.level}"
    if level_key not in baseline_data:
        return fallback_runtime
    # Try exact filename match
    filename = os.path.basename(problem.path) if hasattr(problem, "path") and problem.path else None
    if filename and filename in baseline_data[level_key]:
        return baseline_data[level_key][filename]["mean"]
    # Try problem name match
    for key, stats in baseline_data[level_key].items():
        if problem.name in key or key in problem.name:
            return stats["mean"]
    return fallback_runtime


# Load the best available baseline (prefer H100 PCIe if available; adapt to your hardware)
if baseline_files:
    # Use the first available baseline file
    BASELINE_FILE = baseline_files[0]
    baseline_data = load_baseline(BASELINE_FILE)
    # Show a few entries
    l1_keys = list(baseline_data.get("level1", {}).keys())[:3]
    print(f"Loaded baseline: {BASELINE_FILE}")
    print(f"Sample Level 1 entries: {l1_keys}")
    sample_prob = l1_keys[0] if l1_keys else ""
    if sample_prob:
        print(f"  {sample_prob}: {baseline_data['level1'][sample_prob]['mean']:.4f} ms (torch.compile)")
else:
    print("No pre-computed baseline found.")
    print("You can generate one with:")
    print(f"  uv run python {KERNELBENCH_ROOT}/scripts/generate_baseline_time.py level=1 baseline_type=torch_compile")
    baseline_data = {}

In [ ]:
# Show eager vs torch.compile time for a few Level 1 problems
# This illustrates why torch.compile is a tougher bar

if "level1" in baseline_data:
    eager_file = baseline_files[0].replace(
        "baseline_time_torch_compile_inductor_default.json",
        "baseline_time_torch.json"
    )
    if os.path.exists(eager_file):
        eager_data = load_baseline(eager_file)
        print(f"{'Problem':<55} {'Eager (ms)':>12} {'Compile (ms)':>13} {'Speedup':>8}")
        print("-" * 92)
        for prob_name in list(baseline_data["level1"].keys())[:8]:
            t_compile = baseline_data["level1"][prob_name]["mean"]
            t_eager   = eager_data.get("level1", {}).get(prob_name, {}).get("mean", None)
            if t_eager:
                print(f"{prob_name:<55} {t_eager:>12.4f} {t_compile:>13.4f} {t_eager/t_compile:>8.2f}×")
    else:
        print("Eager baseline file not found alongside compile baseline.")

---
## Section 9 — KernelBench Benchmark SOP

Running a rigorous benchmark follows a **three-step pipeline**.  Each step is a
dedicated KernelBench script that can be run from the command line:

```
Step 1 — Generate kernels
    uv run python scripts/generate_samples.py \
        run_name=<tag>_l<level> level=<level> dataset_src=huggingface backend=triton \
        server_type=local server_address=localhost server_port=8000 \
        model_name=<hf_model_id> temperature=0.0 max_tokens=4096 \
        subset="(1, 20)"
    # Output: runs/<tag>_l<level>/level_<l>_problem_<pid>_sample_0_kernel.py

Step 2 — Evaluate correctness & performance
    uv run python scripts/eval_from_generations.py \
        run_name=<tag>_l<level> level=<level> dataset_src=huggingface backend=triton \
        num_correct_trials=5 num_perf_trials=50 subset="(1, 20)"
    # Output: runs/<tag>_l<level>/eval_results.json

Step 3 — Compute benchmark metrics
    uv run python scripts/benchmark_eval_analysis.py \
        run_name=<tag>_l<level> level=<level> hardware=A100_Colab \
        baseline=baseline_time_torch_compile_inductor_default \
        output_file=results_<tag>_l<level>.json
    # Output: JSON with compilation_rate, correctness_rate, geo_mean_speedup, fast_p
```

> **`server_type=local`** expects a running vLLM server at `localhost:8000`.
> Start one with: `vllm serve <model_id> --port 8000 --trust-remote-code`
>
> **Qwen3-8B thinking mode** (`enable_thinking=True`) requires the `/v1/chat/completions`
> endpoint with `chat_template_kwargs`, which is not yet supported by the
> `generate_samples.py` `server_type=local` preset.  In this notebook we use
> vLLM's offline API for generation (demonstrated in Sections 5–8) and then hand
> the saved kernel files to Steps 2 and 3 of the SOP for evaluation — keeping
> evaluation identical across all model variants.

### Kernel file contract
`eval_from_generations.py` simply looks for files at the path:
```
runs/<run_name>/level_<level>_problem_<pid>_sample_0_kernel.py
```
As long as your generation step writes files there (whether from the CLI script,
from vLLM offline, or from any other inference engine), Steps 2 and 3 work unchanged.
If you save kernels outside `KernelBench/runs/`, pass `runs_dir=<path_to_runs_dir>`
to `eval_from_generations.py` so the evaluator looks in the right directory.

In [ ]:
# ── Benchmark configuration ───────────────────────────────────────────────────
import os, sys, subprocess, json, re
from pathlib import Path
import numpy as np
import pandas as pd


def detect_kernelbench_root() -> Path:
    candidates: list[Path] = []
    env_root = os.getenv("KERNELBENCH_ROOT")
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([
        Path("KernelBench"),
        Path("kernelbench"),
        Path("../KernelBench"),
        Path("../kernelbench"),
    ])
    for candidate in candidates:
        if (candidate / "scripts" / "eval_from_generations.py").exists() and (candidate / "src" / "kernelbench").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find a full KernelBench checkout. Clone it at ./KernelBench, ./kernelbench, ../KernelBench, or ../kernelbench, "
        "or set KERNELBENCH_ROOT to its path before running the benchmark sections."
    )


KERNELBENCH_ROOT = detect_kernelbench_root()

# Add KernelBench scripts to path so we can import analyze_greedy_eval directly
for extra_path in [str(KERNELBENCH_ROOT / "scripts"), str(KERNELBENCH_ROOT / "src")]:
    if extra_path not in sys.path:
        sys.path.insert(0, extra_path)

N_PROBLEMS_PER_LEVEL = 20   # Set to None to run ALL (100 per level)
LEVELS               = [1, 2]
HARDWARE             = os.getenv("KB_HARDWARE", "A100_Colab")
RUNS_DIR             = str(KERNELBENCH_ROOT / "runs")
NUM_CORRECT_TRIALS   = 5
NUM_PERF_TRIALS      = 50   # use 100 for publication-quality timing

BASELINE            = "baseline_time_torch_compile_inductor_default"
BASELINE_JSON       = str(KERNELBENCH_ROOT / "results" / "timing" / HARDWARE / f"{BASELINE}.json")

# Stable run-name stem for each model tag (level is appended later)
RUN_NAME_TAGS = {
    "Qwen2.5-Coder-7B": "qwen25_coder_7b",
    "Qwen3-8B-think":   "qwen3_8b_think",
    "Qwen3-8B-no-think":"qwen3_8b_nothink",
}

def run_name_for(model_tag: str, level: int) -> str:
    return f"{RUN_NAME_TAGS[model_tag]}_l{level}"


def sampling_params_to_dict(sp: SamplingParams) -> dict:
    keys = ["temperature", "top_p", "top_k", "max_tokens", "seed"]
    return {key: getattr(sp, key, None) for key in keys}


def canonicalize_metadata(meta: dict) -> dict:
    normalized = dict(meta)
    if "problem_ids" in normalized:
        normalized["problem_ids"] = sorted(normalized["problem_ids"])
    return normalized


def ensure_json_metadata(path: str, expected: dict) -> None:
    expected = canonicalize_metadata(expected)
    if os.path.exists(path):
        with open(path) as f:
            actual = canonicalize_metadata(json.load(f))
        if actual != expected:
            raise RuntimeError(
                f"Metadata mismatch for {path}. Remove the existing run directory before resuming."
            )
        return
    with open(path, "w") as f:
        json.dump(expected, f, indent=2, sort_keys=True)


def eval_results_complete(eval_path: str, expected_pids: list[int]) -> bool:
    if not os.path.exists(eval_path):
        return False
    try:
        with open(eval_path) as f:
            eval_results = json.load(f)
    except Exception:
        return False
    found_pids = {int(pid_str) for pid_str in eval_results.keys()}
    return set(expected_pids).issubset(found_pids)

ALL_TAGS = list(MODEL_CONFIGS.keys())

print(f"KernelBench root: {KERNELBENCH_ROOT}")
print(f"Config: {N_PROBLEMS_PER_LEVEL} problems × {len(LEVELS)} levels = "
      f"{N_PROBLEMS_PER_LEVEL * len(LEVELS)} problems per model")
print(f"Runs directory : {RUNS_DIR}/")
print(f"Baseline JSON  : {BASELINE_JSON}")
print()
print("Run names:")
for tag, stem in RUN_NAME_TAGS.items():
    print(f"  {tag:<24} → {stem}_l<level>")


In [ ]:
# ── Import the official analysis function from KernelBench scripts ────────────
import importlib.util, pathlib

analysis_script = KERNELBENCH_ROOT / "scripts" / "benchmark_eval_analysis.py"

# Load benchmark_eval_analysis as a module (it lives in scripts/, not a package)
_spec = importlib.util.spec_from_file_location(
    "benchmark_eval_analysis",
    str(analysis_script),
)
_bea = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_bea)
analyze_greedy_eval = _bea.analyze_greedy_eval

print(f"analyze_greedy_eval loaded from {analysis_script}")
print()
import inspect
print(inspect.signature(analyze_greedy_eval))
print()
print("Returns dict with keys: run_name, level, hardware, total_count,")
print("  compiled_count, correct_count, compilation_rate, correctness_rate,")
print("  geo_mean_speedup, fast_p {0.0, 0.5, 0.8, 1.0, 1.5, 2.0}")


In [ ]:
# ── Torch.compile baseline ────────────────────────────────────────────────────
# The baseline measures how long torch.compile takes on every Level 1/2 problem.
# It is a ONE-TIME step per hardware.  If it already exists, skip the command.
#
# To generate (takes ~30 min on A100):
#   !cd KernelBench && uv run python scripts/generate_baseline_time.py
#     (edit HARDWARE_NAME in the script to "A100_Colab" first)
#
# A pre-computed baseline for H100/A100 may be available in:
#   KernelBench/results/timing/

available = sorted((KERNELBENCH_ROOT / "results" / "timing").rglob("*.json"))
if available:
    print("Pre-computed baselines found:")
    for p in available:
        print(f"  {p}")
else:
    print("No pre-computed baselines found.")
    print(f"Run: cd {KERNELBENCH_ROOT} && uv run python scripts/generate_baseline_time.py")

# Load the baseline (needed for speedup computation in Section 11)
if os.path.exists(BASELINE_JSON):
    with open(BASELINE_JSON) as f:
        baseline_data = json.load(f)
    # Flatten: baseline_data["level1"]["problem_name"] = {"mean": ms, ...}
    print(f"\nBaseline loaded from {BASELINE_JSON}")
    print(f"  Levels present: {list(baseline_data.keys())}")
else:
    baseline_data = {}
    print(f"\nWARNING: {BASELINE_JSON} not found.")
    print("Speedup metrics will not be available until the baseline is generated.")

---
## Section 10 — Running the Full Benchmark

The cell below runs the full two-phase pipeline for all model configurations.

**Phase 1 — Generate** (LLM loaded): for each model, generate Triton kernels for every
problem and write them to `runs/<run_name>/`.
- Standard models use the vLLM **offline API** (already loaded in memory).
- Qwen3-8B-think uses `llm.chat()` with `enable_thinking=True` to produce `<think>` blocks.

**Phase 2 — Evaluate** (LLM unloaded): call `eval_from_generations.py` via subprocess.
The evaluation script compiles each kernel, checks correctness, and measures timing
— all without loading the LLM, so the full GPU is available for Triton compilation.

**Approximate wall-clock time on A100 40 GB** (`N_PROBLEMS_PER_LEVEL = 20`):

| Phase | Time per model |
|-------|---------------|
| Generation | 5–15 min (depends on output length) |
| Evaluation (5 correctness + 50 perf trials) | 20–40 min |

Total for the three configurations above: **~2–3 hours**.

> **Tip:** Set `N_PROBLEMS_PER_LEVEL = 5` for a quick dry-run (~20 min total).

In [ ]:
# ── Phase 1: Generate kernels for all models ──────────────────────────────────
from kernelbench.utils import extract_first_code
from kernelbench.prompt_constructor_toml import get_prompt_for_backend
from kernelbench.dataset import construct_kernelbench_dataset

def generate_kernels_for_model(model_tag: str, llm) -> None:
    """
    Generate Triton kernels for all problems in LEVELS using the provided LLM.
    Saves each kernel to:
        {RUNS_DIR}/{run_name}/level_{level}_problem_{pid}_sample_0_kernel.py
    Skips problems that already have a saved kernel (resume-friendly).
    """
    cfg         = MODEL_CONFIGS[model_tag]
    sp          = cfg["sampling_params"]
    thinking    = cfg["thinking"]
    chat_kwargs = cfg.get("chat_template_kwargs")

    for level in LEVELS:
        dataset = construct_kernelbench_dataset(level, source="huggingface")

        # Determine which problem IDs to run
        all_pids = dataset.get_problem_ids()
        if N_PROBLEMS_PER_LEVEL is not None:
            pids = sorted(all_pids)[:N_PROBLEMS_PER_LEVEL]
        else:
            pids = sorted(all_pids)

        run_name = run_name_for(model_tag, level)
        run_dir = os.path.join(RUNS_DIR, run_name)
        os.makedirs(run_dir, exist_ok=True)
        ensure_json_metadata(
            os.path.join(run_dir, "run_meta.json"),
            {
                "model_tag": model_tag,
                "model_id": cfg["model_id"],
                "level": level,
                "thinking": thinking,
                "sampling_params": sampling_params_to_dict(sp),
                "prompt_option": "one_shot",
                "problem_ids": sorted(pids),
            },
        )

        print(f"  Generating L{level}: {len(pids)} problems → {run_dir}/")

        for pid in pids:
            out_path = os.path.join(
                run_dir,
                f"level_{level}_problem_{pid}_sample_0_kernel.py"
            )
            if os.path.exists(out_path):
                continue  # already generated — resume-friendly

            problem = dataset.get_problem_by_id(pid)
            prompt = get_prompt_for_backend(ref_arch_src=problem.code, backend="triton")

            messages = [{"role": "user", "content": prompt}]
            if chat_kwargs:
                outputs = llm.chat(
                    [messages],
                    sampling_params=sp,
                    chat_template_kwargs=chat_kwargs,
                )
            else:
                outputs = llm.chat([messages], sampling_params=sp)

            raw = outputs[0].outputs[0].text
            if thinking:
                raw = strip_think_block(raw)

            code = extract_first_code(raw, ["python", "py"])
            with open(out_path, "w") as fh:
                fh.write(code if code else raw)

        print("    done.")


# ── Run generation for all model groups (weight-sharing for Qwen3-8B) ─────────
from collections import defaultdict

tags_by_model_id = defaultdict(list)
for tag in MODEL_CONFIGS:
    tags_by_model_id[MODEL_CONFIGS[tag]["model_id"]].append(tag)

for model_id, tags in tags_by_model_id.items():
    print("=" * 70)
    print(f"Loading: {model_id}  →  tags: {tags}")
    llm = load_vllm_model(tags[0])

    for tag in tags:
        print(f"  Generating kernels for [{tag}]")
        generate_kernels_for_model(tag, llm)

    unload_vllm_model(llm)
    print()

print("Phase 1 complete — all kernel files saved to", RUNS_DIR)


In [ ]:
# ── Phase 2: Evaluate kernels using the KernelBench CLI script ───────────────
# eval_from_generations.py reads the kernel files saved in Phase 1 and writes
# runs/{run_name}/eval_results.json  (keyed by problem_id string)

def run_eval_script(run_name: str, level: int) -> bool:
    """Run eval_from_generations.py for one (run_name, level) pair via subprocess."""
    cmd = [
        "uv", "run", "python", str(KERNELBENCH_ROOT / "scripts" / "eval_from_generations.py"),
        f"run_name={run_name}",
        f"level={level}",
        "dataset_src=huggingface",
        "backend=triton",
        f"num_correct_trials={NUM_CORRECT_TRIALS}",
        f"num_perf_trials={NUM_PERF_TRIALS}",
        f"runs_dir={RUNS_DIR}",
    ]
    if N_PROBLEMS_PER_LEVEL is not None:
        cmd.append(f"subset=(1,{N_PROBLEMS_PER_LEVEL})")

    print(f"  Evaluating {run_name} / level {level} ...")
    result = subprocess.run(cmd, capture_output=False, text=True)
    if result.returncode != 0:
        raise RuntimeError(
            f"eval_from_generations.py failed for {run_name} / level {level} with exit code {result.returncode}"
        )
    expected_pids = sorted(construct_kernelbench_dataset(level, source="huggingface").get_problem_ids())
    if N_PROBLEMS_PER_LEVEL is not None:
        expected_pids = expected_pids[:N_PROBLEMS_PER_LEVEL]
    eval_json = os.path.join(RUNS_DIR, run_name, "eval_results.json")
    if not eval_results_complete(eval_json, expected_pids):
        raise RuntimeError(f"Incomplete eval results for {run_name} / level {level}")
    return True


for tag in ALL_TAGS:
    print(f"\n[{tag}]")
    for level in LEVELS:
        run_name = run_name_for(tag, level)
        eval_json = os.path.join(RUNS_DIR, run_name, "eval_results.json")
        expected_pids = sorted(construct_kernelbench_dataset(level, source="huggingface").get_problem_ids())
        if N_PROBLEMS_PER_LEVEL is not None:
            expected_pids = expected_pids[:N_PROBLEMS_PER_LEVEL]
        if eval_results_complete(eval_json, expected_pids):
            print(f"  Level {level}: complete eval_results.json already exists — skipping")
            continue
        run_eval_script(run_name, level)

print("\nPhase 2 complete — eval_results.json written for each run.")


---
## Section 11 — Results Analysis & Visualisation

Now we analyse and visualise the benchmark results across all model configurations.

Results are loaded from the `eval_results.json` files written by the evaluation
script.  We also run the custom `check_triton_validity()` checker on each saved
kernel file to compute the **validity** metric.

In [ ]:
# ── 11.0  Load per-problem DataFrame and build summary table ─────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from kernelbench.dataset import construct_kernelbench_dataset

# --------------------------------------------------------------------------
# Helper: load eval_results.json + validity check → list of row dicts
# --------------------------------------------------------------------------
def load_run_results(model_tag: str) -> list[dict]:
    rows = []

    for level in LEVELS:
        run_name = run_name_for(model_tag, level)
        eval_path = os.path.join(RUNS_DIR, run_name, "eval_results.json")
        if not os.path.exists(eval_path):
            print(f"  WARNING: {eval_path} not found — skipping level {level}")
            continue

        with open(eval_path) as f:
            eval_results = json.load(f)

        # Build problem_name → baseline_time mapping
        baseline_for_level: dict[str, float] = {}
        if baseline_data:
            level_key = f"level{level}"
            for prob_name, bdata in baseline_data.get(level_key, {}).items():
                baseline_for_level[prob_name] = bdata.get("mean", None)

        # Get problem names from dataset (pid → name)
        dataset = construct_kernelbench_dataset(level, source="huggingface")
        pid_to_name = {pid: dataset.get_problem_by_id(pid).name
                       for pid in dataset.get_problem_ids()}

        for pid_str, entries in eval_results.items():
            pid = int(pid_str)
            entry = next((e for e in entries if e["sample_id"] == 0), None)
            if entry is None:
                continue

            # Validity check from saved kernel file
            kpath = os.path.join(
                RUNS_DIR, run_name, f"level_{level}_problem_{pid}_sample_0_kernel.py"
            )
            kernel_generated = os.path.exists(kpath)
            if kernel_generated:
                validity = check_triton_validity(open(kpath).read())
            else:
                validity = {"is_valid": False, "needs_manual_review": False}

            # Speedup vs torch.compile
            prob_name   = pid_to_name.get(pid, "")
            baseline_ms = baseline_for_level.get(prob_name)
            actual_ms   = entry.get("runtime")
            speedup     = (
                baseline_ms / actual_ms
                if baseline_ms and actual_ms and actual_ms > 0 else None
            )

            rows.append({
                "model":               model_tag,
                "level":               level,
                "problem_id":          pid,
                "problem_name":        prob_name,
                "problem":             f"L{level}_P{pid:03d}",
                "compiled":            bool(entry.get("compiled", False)),
                "correctness":         bool(entry.get("correctness", False)),
                "is_valid":            bool(validity.get("is_valid", False)),
                "needs_manual_review": bool(validity.get("needs_manual_review", False)),
                "runtime_ms":          actual_ms,
                "baseline_ms":         baseline_ms,
                "speedup_vs_compile":  speedup,
                "kernel_generated":    kernel_generated,
            })
    return rows


# Build per-problem DataFrame
all_rows = []
for tag in ALL_TAGS:
    all_rows.extend(load_run_results(tag))

df = pd.DataFrame(all_rows)
print(f"Total rows: {len(df)}")
df.head(3)


In [ ]:
# ── 11.1  Summary table from per-problem rows ────────────────────────────────
from kernelbench.score import fastp, geometric_mean_speed_ratio_correct_only

summary_rows = {}

for tag in ALL_TAGS:
    tag_rows = df[df["model"] == tag].copy()
    if tag_rows.empty:
        continue

    aligned = tag_rows[
        tag_rows["baseline_ms"].notna() & tag_rows["runtime_ms"].notna()
    ].copy()

    n_eval = len(tag_rows)
    n_aligned = len(aligned)

    if n_aligned > 0:
        is_correct = aligned["correctness"].to_numpy(dtype=bool)
        baseline_speed = aligned["baseline_ms"].to_numpy(dtype=float)
        actual_speed = aligned["runtime_ms"].to_numpy(dtype=float)
        geo_mean_speedup = float(
            geometric_mean_speed_ratio_correct_only(
                is_correct, baseline_speed, actual_speed, n_aligned
            )
        )
        fast_0 = 100 * fastp(is_correct, baseline_speed, actual_speed, n_aligned, 0.0)
        fast_1 = 100 * fastp(is_correct, baseline_speed, actual_speed, n_aligned, 1.0)
        fast_2 = 100 * fastp(is_correct, baseline_speed, actual_speed, n_aligned, 2.0)
    else:
        geo_mean_speedup = 0.0
        fast_0 = fast_1 = fast_2 = 0.0

    summary_rows[tag] = {
        "Evaluated":        n_eval,
        "Compiled %":       tag_rows["compiled"].mean() * 100,
        "Correct %":        tag_rows["correctness"].mean() * 100,
        "Valid %":          tag_rows["is_valid"].mean() * 100,
        "GeoMean Speedup":  geo_mean_speedup,
        "fast_0":           fast_0,
        "fast_1":           fast_1,
        "fast_2":           fast_2,
    }

summary_df = pd.DataFrame(summary_rows).T

fmt = {
    "Evaluated": "{:.0f}",
    "Compiled %": "{:.1f}%",
    "Correct %": "{:.1f}%",
    "Valid %": "{:.1f}%",
    "GeoMean Speedup": "{:.3f}×",
    "fast_0": "{:.1f}%",
    "fast_1": "{:.1f}%",
    "fast_2": "{:.1f}%",
}
print("=== Summary Table ===")
display(summary_df.style.format(fmt))


In [ ]:
# ── 11.2  Grouped bar chart: Compilation / Correctness / Validity ──────────
sns.set_theme(style="whitegrid", font_scale=1.1)

fig, ax = plt.subplots(figsize=(9, 5))

metrics     = ["Compiled %", "Correct %", "Valid %"]
x           = np.arange(len(metrics))
n_models    = len(summary_df)
bar_width   = 0.22
colors      = sns.color_palette("Set2", n_models)

for i, (model_name, row) in enumerate(summary_df.iterrows()):
    offset = (i - n_models / 2 + 0.5) * bar_width
    vals   = [row[m] for m in metrics]
    bars   = ax.bar(x + offset, vals, bar_width, label=model_name, color=colors[i])
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{v:.0f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(["Compilation\nRate", "Correctness\nRate", "Validity\nRate"])
ax.set_ylabel("Percentage (%)")
ax.set_ylim(0, 115)
ax.set_title("Compilation, Correctness, and Validity Rates\n"
             "(Triton backend, vs torch.compile, A100 40 GB)")
ax.legend(loc="upper right")
plt.tight_layout()
os.makedirs(RUNS_DIR, exist_ok=True)
plt.savefig(os.path.join(RUNS_DIR, "fig_rates.png"), dpi=150)
plt.show()

In [ ]:
# ── 11.3  fast_p bar chart ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

fastp_metrics = ["fast_0", "fast_1", "fast_2"]
x             = np.arange(len(fastp_metrics))

for i, (model_name, row) in enumerate(summary_df.iterrows()):
    offset = (i - n_models / 2 + 0.5) * bar_width
    vals   = [row[m] for m in fastp_metrics]
    bars   = ax.bar(x + offset, vals, bar_width, label=model_name, color=colors[i])
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{v:.0f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([
    "fast_0\n(correct)",
    "fast_1\n(>1× torch.compile)",
    "fast_2\n(>2× torch.compile)",
])
ax.set_ylabel("% of problems")
ax.set_ylim(0, 115)
ax.set_title("fast_p Scores — Fraction of Problems Correct AND Faster than torch.compile")
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig(os.path.join(RUNS_DIR, "fig_fast_p.png"), dpi=150)
plt.show()

In [ ]:
# ── 11.4  Speedup distribution (violin plot, log scale) ────────────────────
df_correct = df[
    df["correctness"] &
    df["speedup_vs_compile"].notna() &
    (df["speedup_vs_compile"] > 0)
].copy()
df_correct["log_speedup"] = np.log2(df_correct["speedup_vs_compile"])

fig, ax = plt.subplots(figsize=(9, 5))

if len(df_correct) > 0:
    sns.violinplot(
        data=df_correct, x="model", y="log_speedup",
        palette="Set2", inner="box", cut=0, ax=ax
    )
    ax.axhline(0, color="red", linestyle="--", linewidth=1.5,
               label="1× (same as torch.compile)")
    ax.set_ylabel("Speedup vs torch.compile  (log₂ scale)")
    ax.set_xlabel("")
    yticks = ax.get_yticks()
    ax.set_yticklabels([f"{2**t:.2f}×" for t in yticks])
    ax.set_title("Speedup Distribution for Correct Kernels\n(log₂ scale)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(RUNS_DIR, "fig_speedup_violin.png"), dpi=150)
    plt.show()
else:
    print("No correct kernels with speedup data to plot.")

In [ ]:
# ── 11.5  Level breakdown: L1 vs L2 correctness ────────────────────────────
df_level = (
    df.groupby(["model", "level"])["correctness"]
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={"correctness": "Correct %"})
)
df_level["Level"] = df_level["level"].map({1: "Level 1\n(single-kernel)",
                                            2: "Level 2\n(fusion patterns)"})

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(
    data=df_level, x="Level", y="Correct %", hue="model",
    palette="Set2", ax=ax
)
ax.set_ylim(0, 115)
ax.set_title("Correctness Rate by Level\n(Level 2 fusion is harder)")
ax.set_xlabel("")
ax.legend(title="Model")
plt.tight_layout()
plt.savefig(os.path.join(RUNS_DIR, "fig_level_breakdown.png"), dpi=150)
plt.show()

In [ ]:
# ── 11.6  Validity vs Correctness scatter ──────────────────────────────────
vd = (
    df.groupby("model")[["is_valid", "correctness"]]
    .mean()
    .mul(100)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(6, 6))
for i, row in vd.iterrows():
    ax.scatter(row["is_valid"], row["correctness"], s=180, color=colors[i],
               label=row["model"], zorder=5)
    ax.annotate(row["model"], (row["is_valid"], row["correctness"]),
                textcoords="offset points", xytext=(8, 4), fontsize=9)

ax.plot([0, 100], [0, 100], "k--", linewidth=0.8, label="valid = correct")
ax.set_xlim(0, 105); ax.set_ylim(0, 105)
ax.set_xlabel("Validity Rate (%)"); ax.set_ylabel("Correctness Rate (%)")
ax.set_title("Validity vs Correctness\n(above diagonal: valid but wrong)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(RUNS_DIR, "fig_validity_vs_correctness.png"), dpi=150)
plt.show()

In [ ]:
# ── 11.7  Problem-level correctness heatmap ─────────────────────────────────
heatmap_df = df.pivot_table(
    index="problem",
    columns="model",
    values="correctness",
    aggfunc="max",
    fill_value=0,
).astype(float)

heatmap_df = heatmap_df.assign(_total=heatmap_df.sum(axis=1)).sort_values(
    "_total", ascending=False
).drop(columns="_total")

fig_h = max(8, len(heatmap_df) * 0.35)
fig, ax = plt.subplots(figsize=(7, fig_h))
sns.heatmap(
    heatmap_df, cmap="RdYlGn", linewidths=0.3, vmin=0, vmax=1,
    cbar_kws={"label": "Correct (1) / Wrong (0)"}, ax=ax,
)
ax.set_title("Per-Problem Correctness Heatmap", pad=12)
ax.set_xlabel(""); ax.set_ylabel("")
ax.tick_params(axis="y", labelsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RUNS_DIR, "fig_heatmap.png"), dpi=150)
plt.show()

In [ ]:
# ── 11.8  Manual review table — borderline validity cases ───────────────────
review_df = df[
    df["needs_manual_review"] & df["kernel_generated"]
][["model", "level", "problem", "is_valid", "correctness"]]

if len(review_df) > 0:
    print(f"{len(review_df)} kernels flagged for manual review (torch.* inside @triton.jit):")
    display(review_df.reset_index(drop=True))
    print("\nKernel source files are in:", RUNS_DIR)
else:
    print("No kernels flagged for manual review.")

# ── 11.9  Print final summary table ────────────────────────────────────────
print("\n" + "=" * 75)
print("BENCHMARK SUMMARY (Triton backend, torch.compile baseline, A100 40 GB)")
print("=" * 75)
print(summary_df.to_string(float_format="{:.1f}".format))

---
## Key Findings & Next Steps

### Interpreting the metrics

| Metric | What it means |
|--------|---------------|
| **Compilation rate** | Can the LLM write syntactically valid Triton code? |
| **Correctness rate** | Does the kernel produce the right numerical answer? |
| **Validity rate** | Did the LLM _actually_ write a Triton kernel (not just wrap PyTorch)? |
| **fast_0** | Same as correctness — the easiest bar. |
| **fast_1** | Correct AND faster than `torch.compile` — a meaningful engineering win. |
| **fast_2** | Correct AND 2× faster — only achievable with true fusion or algorithmic insight. |
| **GeoMean Speedup** | Geometric mean speedup across all _correct_ kernels. |

### Suggested follow-ups

1. **Scale up** — run all 200 problems (Level 1 + 2) for statistically robust results.
2. **More models** — try larger models (e.g. `Qwen/Qwen2.5-Coder-32B-Instruct` via quantisation) or reasoning models.
3. **Iterative refinement** — feed correctness/profiling feedback back to the LLM (see [Caesar](https://github.com/ScalingIntelligence/caesar)).
4. **Custom validity** — extend `check_triton_validity()` to also detect autotune decorators and tl.constexpr usage, which signal more sophisticated kernels.
5. **Level 3** — try full model architectures for a much harder challenge.

---

### Adapting to a production server setup

This notebook used vLLM in **offline mode** for simplicity. For multi-user / production use, run vLLM as a server and point KernelBench at it:

```bash
# Terminal 1: start vLLM server
python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-Coder-7B-Instruct \
    --port 8000

# Terminal 2: run KernelBench generation
uv run python KernelBench/scripts/generate_samples.py \
    run_name=qwen_triton_l1 dataset_src=huggingface level=1 \
    server_type=local server_address=localhost server_port=8000 \
    model_name=Qwen/Qwen2.5-Coder-7B-Instruct backend=triton
```

---

### Citation

```bibtex
@misc{ouyang2025kernelbench,
  title   = {KernelBench: Can LLMs Write Efficient GPU Kernels?},
  author  = {Anne Ouyang and Simon Guo and Simran Arora and Alex L. Zhang
             and William Hu and Christopher Ré and Azalia Mirhoseini},
  year    = {2025},
  url     = {https://arxiv.org/abs/2502.10517}
}
```